## 1. Pr?paration de l?environnement local ou Google Colab (YOLOv9)

Dans Colab, s?lectionnez **Runtime > Change runtime type > GPU** avant l?ex?cution. Ce notebook v?rifie ensuite CUDA; le code ne peut pas activer le GPU ? votre place. Le dataset existant est attendu dans `../trafic_data`.


In [ ]:
# Décommenter si les dépendances ne sont pas encore installées :
# %pip install -r ../../requirements.txt

import sys
import torch
import torchvision
import cv2
import ultralytics
import numpy as np
import pandas as pd
import matplotlib
import yaml
import sklearn
import PIL
from ultralytics import YOLO

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("CUDA dispo. :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
else:
    print("GPU absent : dans Colab, s?lectionnez Runtime > Change runtime type > GPU")
print("Torchvision :", torchvision.__version__)
print("OpenCV      :", cv2.__version__)
print("Ultralytics :", ultralytics.__version__)
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("Matplotlib  :", matplotlib.__version__)
print("PyYAML      :", yaml.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Pillow      :", PIL.__version__)

In [ ]:
# Vérification de l'environnement Ultralytics et de l'architecture YOLOv9
ultralytics.checks()

model_yolov9 = YOLO("yolov9c.yaml")
print("Architecture YOLOv9 prête :", type(model_yolov9.model).__name__)

## 2. Exploration du dataset

In [ ]:
from pathlib import Path
import random
import yaml
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# Chemin du dataset
DATASET_DIR = Path("../trafic_data")

# Vérification
print("Dataset accessible :", DATASET_DIR.exists())
print("Chemin du dataset :", DATASET_DIR.resolve())

In [ ]:
# Afficher les dossiers et fichiers à la racine du dataset

for element in sorted(DATASET_DIR.iterdir()):
    print(element.name, ":", "Dossier" if element.is_dir() else "Fichier")

In [ ]:
# Extensions d'images acceptées
EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

for split in ["train", "valid", "test"]:
    dossier_images = DATASET_DIR / split / "images"
    dossier_labels = DATASET_DIR / split / "labels"

    images = [
        f for f in dossier_images.iterdir()
        if f.is_file() and f.suffix.lower() in EXTENSIONS
    ]

    labels = list(dossier_labels.glob("*.txt"))

    print(f"\n--- {split.upper()} ---")
    print("Nombre d'images :", len(images))
    print("Nombre de Labels :", len(labels))

In [ ]:
# Récupérer les images du dossier train
dossier_images = DATASET_DIR / "train" / "images"

images_train = sorted([
    f for f in dossier_images.iterdir()
    if f.is_file() and f.suffix.lower() in EXTENSIONS
])

# Afficher une image
if images_train:
    image_path = images_train[0]

    with Image.open(image_path) as image:
        plt.figure(figsize=(8, 6))
        plt.imshow(image.copy())
        plt.title(image_path.name)
        plt.axis("off")
        plt.show()
else:
    print("Aucune image trouvée.")

## 4. Installation et configuration de YOLOv9

Le d?p?t officiel est `WongKinYiu/yolov9`. Dans un environnement neuf, depuis la racine du d?p?t, installez ses d?pendances avec `python -m pip install "setuptools<81" -r requirements.txt` (cette version conserve `pkg_resources`, utilis? par ce d?p?t). Le notebook clone le d?p?t s?il manque et r?cup?re les poids officiels seulement s?ils ne sont pas pr?sents.

In [ ]:
import subprocess
import sys
from urllib.request import urlretrieve

YOLO_REPO_URL = "https://github.com/WongKinYiu/yolov9.git"
YOLO_ROOT = (DATASET_DIR.parent / "yolov9").resolve()
if not YOLO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", YOLO_REPO_URL, str(YOLO_ROOT)], check=True)

remote = subprocess.run(
    ["git", "-C", str(YOLO_ROOT), "remote", "get-url", "origin"],
    check=True, capture_output=True, text=True,
).stdout.strip().rstrip("/")
if remote.removesuffix(".git") != YOLO_REPO_URL.removesuffix(".git"):
    raise ValueError(f"D?p?t inattendu : {remote}")

TRAIN_SCRIPT = YOLO_ROOT / "train_dual.py"
MODEL_CONFIG = YOLO_ROOT / "models" / "detect" / "yolov9-c.yaml"
HYP_CONFIG = YOLO_ROOT / "data" / "hyps" / "hyp.scratch-high.yaml"
WEIGHTS_PATH = YOLO_ROOT / "yolov9-c.pt"
DATA_CONFIG = DATASET_DIR / "data_1.yaml"
WEIGHTS_URL = "https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt"
OFFICIAL_WEIGHTS_SIZE = 103153312

if not WEIGHTS_PATH.exists():
    urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
if WEIGHTS_PATH.stat().st_size != OFFICIAL_WEIGHTS_SIZE:
    raise ValueError(f"Taille inattendue pour {WEIGHTS_PATH.name}: {WEIGHTS_PATH.stat().st_size} octets")

for required_path in [TRAIN_SCRIPT, MODEL_CONFIG, HYP_CONFIG, WEIGHTS_PATH, DATA_CONFIG]:
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

print("D?p?t officiel :", remote)
print("Script entra?nement :", TRAIN_SCRIPT)
print("Architecture :", MODEL_CONFIG)
print("Poids pr?entra?n?s :", WEIGHTS_PATH, f"({WEIGHTS_PATH.stat().st_size:,} octets)")
print("Hyperparam?tres :", HYP_CONFIG)
print("Dataset :", DATA_CONFIG)


### R?le des fichiers et Transfer Learning

- `models/detect/yolov9-c.yaml` d?crit l?architecture (couches, connexions et t?tes de d?tection), pas les valeurs apprises. Il d?clare 80 classes par d?faut.
- `yolov9-c.pt` contient les poids appris du mod?le officiel pr?entra?n?; il ne remplace ni la configuration d?architecture ni le dataset.
- `train_dual.py` construit le mod?le demand? avec le nombre de classes du dataset (21), charge les tenseurs compatibles du checkpoint et ignore les tenseurs dont les dimensions ne correspondent pas, notamment la t?te de d?tection adapt?e aux classes. Il affiche le nombre de tenseurs transf?r?s.
- `data/hyps/hyp.scratch-high.yaml` r?gle l?optimisation et les augmentations (taux d?apprentissage, momentum, poids de pertes, mosa?que, etc.).

Le Transfer Learning r?utilise donc les repr?sentations visuelles d?j? apprises, puis ajuste le mod?le aux cat?gories de v?hicules du dataset. L?architecture d?finit la structure, les poids d?finissent son ?tat appris et le script d?finit la proc?dure d?entra?nement.

In [ ]:
import shlex
import subprocess
import yaml

model_cfg = yaml.safe_load(MODEL_CONFIG.read_text(encoding="utf-8"))
data_cfg = yaml.safe_load(DATA_CONFIG.read_text(encoding="utf-8"))
hyp_cfg = yaml.safe_load(HYP_CONFIG.read_text(encoding="utf-8"))

assert model_cfg["nc"] == 80, f"Architecture attendue ? 80 classes, trouv?: {model_cfg['nc']}"
assert data_cfg["nc"] == len(data_cfg["names"]) == 21
resolved_dataset = (YOLO_ROOT / data_cfg["path"]).resolve()
assert resolved_dataset == DATASET_DIR.resolve(), (resolved_dataset, DATASET_DIR.resolve())
for key, split in [("train", "train"), ("val", "valid"), ("test", "test")]:
    image_dir = (resolved_dataset / data_cfg[key]).resolve()
    assert image_dir.is_dir(), f"Chemin {key} introuvable : {image_dir}"
    print(f"{key:5s} -> {image_dir}")

print("Classes du mod?le de base :", model_cfg["nc"])
print("Classes du dataset :", data_cfg["nc"])
print("Hyperparam?tres cl?s :", {key: hyp_cfg[key] for key in ["lr0", "momentum", "weight_decay", "mosaic"]})

help_result = subprocess.run(
    [sys.executable, str(TRAIN_SCRIPT), "--help"],
    cwd=YOLO_ROOT, check=True, capture_output=True, text=True,
)
print("CLI entra?nement :", help_result.stdout.splitlines()[0])

training_command = [
    sys.executable, str(TRAIN_SCRIPT),
    "--workers", "2", "--device", "0", "--batch-size", "8",
    "--data", str(DATA_CONFIG), "--img", "640",
    "--cfg", str(MODEL_CONFIG), "--weights", str(WEIGHTS_PATH),
    "--name", "trafficsight-yolov9-c", "--hyp", str(HYP_CONFIG),
    "--min-items", "0", "--epochs", "100", "--close-mosaic", "15",
]
print("Commande de d?part (? ex?cuter depuis YOLO_ROOT, GPU requis) :")
print(subprocess.list2cmdline(training_command) if sys.platform == "win32" else shlex.join(training_command))
